Claude Sonnet 4.6 was used to assist me in writing and debugging the complex scripts required to properly scrape minutes text and assign appropriate labels for training downstream.

In this notebook, we iteratively scrape minutes text for BoE and RBA from the following sources, and append to central_bank_minutes.csv:

  - BoE: bankofengland.co.uk (PDF)
  - RBA: rba.gov.au (HTML)

We then append rates for each central bank and assign labels:

  - FOMC: hardcoded (FRED API not working from Colab)
  - BoE: via BOEBR.csv downloaded from bankofengland.co.uk
  - RBA: directly from rba.gov.au A2 table, with hardcoding as a fallback option

Labels are assigned by comparing rate change delta between current and prior periods to determine hold versus change class.

After labeling, we validate central_bank_minutes_labeled.csv with several confirmed rate change periods to ensure accuracy.

Please see central_bank_minutes_labeled.csv for final output used for the remainder of the project.

In [ ]:
# import and install beautifulsoup, tqdm, pdfplumber, and all other packages needed to properly scrape each data source

!pip install datasets requests beautifulsoup4 pandas tqdm pdfplumber -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import requests
import re
import io
import time
from io import StringIO
from bs4 import BeautifulSoup
from tqdm import tqdm
import pdfplumber

# header used for scraper agent

HEADERS = {'User-Agent': 'Mozilla/5.0 (research project)'}

In [ ]:
# upload FOMC minutes dataset from vtasca, no scraping required

from datasets import load_dataset

ds = load_dataset('vtasca/fomc-statements-minutes', split='train')
fomc_df = ds.to_pandas()

type_col    = next(c for c in fomc_df.columns if 'type' in c.lower())
date_col    = next(c for c in fomc_df.columns if c.lower() == 'date' or ('date' in c.lower() and 'release' not in c.lower()))
release_col = next((c for c in fomc_df.columns if 'release' in c.lower()), None)
text_col    = next(c for c in fomc_df.columns if 'text' in c.lower())
minutes_val = next(v for v in fomc_df[type_col].unique() if 'minut' in str(v).lower())

fomc_minutes = fomc_df[fomc_df[type_col] == minutes_val].copy()
fomc_out = pd.DataFrame({
    'bank':         'FOMC'
    , 'meeting_date': pd.to_datetime(fomc_minutes[date_col])
    , 'release_date': pd.to_datetime(fomc_minutes[release_col]) if release_col else pd.NaT
    , 'text':         fomc_minutes[text_col].values
    , 'url':          fomc_minutes['url'].values if 'url' in fomc_minutes.columns else None
})

print(f'FOMC: {len(fomc_out)} documents ({fomc_out["meeting_date"].min().date()} → {fomc_out["meeting_date"].max().date()})')

In [ ]:
# get BoE minutes via bankofengland.co.uk via sitemap and beautifulsoup
# two URL patterns exist pre and post-2016 and have to be handled differently

def get_boe_urls_from_sitemap():
    r = requests.get('https://www.bankofengland.co.uk/sitemap/minutes', headers=HEADERS, timeout=15)
    soup = BeautifulSoup(r.text, 'html.parser')
    urls = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        if ('monetary-policy-summary-and-minutes' in href or
                ('/minutes/' in href and 'monetary-policy-committee' in href)):
            full = 'https://www.bankofengland.co.uk' + href if href.startswith('/') else href
            if not full.endswith('.pdf') and re.search(r'/(\d{4})/[\w-]+$', full):
                urls.append(full)
    return list(set(urls))

# convert html from URLs retrieved to PDFs

def html_url_to_pdf_candidates(html_url):
    m = re.search(r'/(\d{4})/([\w-]+)$', html_url)
    if not m:
        return []
    year, slug = m.group(1), m.group(2)
    base = 'https://www.bankofengland.co.uk/-/media/boe/files'
    if 'monetary-policy-summary-and-minutes' in html_url:
        clean_slug = re.sub(r'^mpc-', '', slug)
        return [
            f'{base}/monetary-policy-summary-and-minutes/{year}/monetary-policy-summary-and-minutes-{clean_slug}.pdf'
            , f'{base}/monetary-policy-summary-and-minutes/{year}/{clean_slug}.pdf'
            , f'{base}/monetary-policy-summary-and-minutes/{year}/minutes-{clean_slug}.pdf'
            , f'{base}/monetary-policy-summary-and-minutes/{year}/{slug}.pdf'
        ]
    else:
        month_slug = re.sub(r'^monetary-policy-committee-', '', slug)
        return [
            f'{base}/minutes/{year}/minutes-{month_slug}.pdf'
            , f'{base}/minutes/{year}/{month_slug}.pdf'
            , f'{base}/monetary-policy-summary-and-minutes/{year}/minutes-{month_slug}.pdf',
        ]

# extract the minutes text from each PDF

def extract_text_from_pdf_url(pdf_url):
    try:
        r = requests.get(pdf_url, headers=HEADERS, timeout=30)
        if r.status_code != 200:
            return None
        with pdfplumber.open(io.BytesIO(r.content)) as pdf:
            text = ' '.join(page.extract_text() or '' for page in pdf.pages)
        text = re.sub(r'\s+', ' ', text).strip()
        return text if len(text) > 500 else None
    except:
        return None

# extract the date from the BoE URL

def extract_date_from_boe_url(url):
    match = re.search(r'([a-z]+)-(\d{4})$', url)
    if match:
        try:
            return pd.to_datetime(f"1 {match.group(1)} {match.group(2)}")
        except:
            return pd.NaT
    return pd.NaT

print('Fetching BoE sitemap...')
boe_html_urls = get_boe_urls_from_sitemap()
print(f'Found {len(boe_html_urls)} URLs. Scraping...')

boe_records = []
for html_url in tqdm(boe_html_urls):
    text, used_url = None, None
    for pdf_url in html_url_to_pdf_candidates(html_url):
        text = extract_text_from_pdf_url(pdf_url)
        if text:
            used_url = pdf_url
            break
    if text:
        boe_records.append({
            'bank': 'BoE', 'meeting_date': extract_date_from_boe_url(html_url),
            'release_date': pd.NaT, 'text': text, 'url': used_url
        })
    time.sleep(0.3)

boe_df = pd.DataFrame(boe_records)
print(f'BoE: {len(boe_df)} documents ({boe_df["meeting_date"].min().date()} → {boe_df["meeting_date"].max().date()})')

In [ ]:
# get RBA meeting minutes via rba.gov.au year index pages using two distinct URL date formats depending on time period

def get_rba_minutes_urls(year):
    try:
        r = requests.get(f'https://www.rba.gov.au/monetary-policy/rba-board-minutes/{year}/'
                         , headers=HEADERS, timeout=15)
        if r.status_code != 200:
            return []
        soup = BeautifulSoup(r.text, 'html.parser')
        links = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            if re.search(r'/rba-board-minutes/\d{4}/(\d{4}-\d{2}-\d{2}|\d{8})\.html', href):
                full = 'https://www.rba.gov.au' + href if href.startswith('/') else href
                links.append(full)
        return list(set(links))
    except:
        return []

# scrape each RBA url using beautifulsoup

def scrape_rba_page(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            return None
        soup = BeautifulSoup(r.text, 'html.parser')
        content = (soup.find('div', id='content') or soup.find('div', class_='content-wrapper')
                   or soup.find('main') or soup.find('article'))
        if not content:
            return None
        for tag in content.find_all(['nav', 'footer', 'aside', 'script', 'style']):
            tag.decompose()
        text = re.sub(r'\s+', ' ', content.get_text(separator=' ', strip=True)).strip()
        return text if len(text) > 500 else None
    except:
        return None

# extract date from HTML output

def extract_date_from_rba_url(url):
    m = re.search(r'(\d{4}-\d{2}-\d{2})\.html', url)
    if m:
        return pd.to_datetime(m.group(1))
    m = re.search(r'/(\d{2})(\d{2})(\d{4})\.html', url)
    if m:
        return pd.to_datetime(f'{m.group(3)}-{m.group(2)}-{m.group(1)}')
    return pd.NaT

print('Collecting RBA URLs...')
all_rba_urls = []
for year in range(2006, 2027):
    all_rba_urls.extend(get_rba_minutes_urls(year))
    time.sleep(0.3)
print(f'Found {len(all_rba_urls)} URLs. Scraping...')

rba_records = []
for url in tqdm(all_rba_urls):
    text = scrape_rba_page(url)
    if text:
        rba_records.append({
            'bank': 'RBA', 'meeting_date': extract_date_from_rba_url(url),
            'release_date': pd.NaT, 'text': text, 'url': url
        })
    time.sleep(0.3)

rba_df = pd.DataFrame(rba_records)
print(f'RBA: {len(rba_df)} documents ({rba_df["meeting_date"].min().date()} → {rba_df["meeting_date"].max().date()})')

In [ ]:
# combine all results into a single dataframe and save results prior to adding labels

minutes_df = pd.concat([fomc_out, boe_df, rba_df], ignore_index=True)
minutes_df['meeting_date'] = pd.to_datetime(minutes_df['meeting_date'])
minutes_df['word_count']   = minutes_df['text'].str.split().str.len()
minutes_df = minutes_df.sort_values(['bank', 'meeting_date']).reset_index(drop=True)

minutes_df.to_csv('/content/drive/MyDrive/central_bank_minutes.csv', index=False)

print(f'Saved {len(minutes_df)} documents → central_bank_minutes.csv')
print(minutes_df.groupby('bank').agg(
    count=('text','count')
    , date_min=('meeting_date','min')
    , date_max=('meeting_date','max')
    , avg_words=('word_count','mean')
).round(0))

In [ ]:
# add FOMC rate decisions hard-coded from downloaded CSV files for simplicity (FRED API unreliable from Google Colab)

FOMC_DECISIONS = [
    ('2000-01-01', 5.50), ('2001-01-03', 6.00), ('2001-01-31', 5.50),
    ('2001-03-20', 5.00), ('2001-04-18', 4.50), ('2001-05-15', 4.00),
    ('2001-06-27', 3.75), ('2001-08-21', 3.50), ('2001-09-17', 3.00),
    ('2001-10-02', 2.50), ('2001-11-06', 2.00), ('2001-12-11', 1.75),
    ('2002-11-06', 1.25), ('2003-06-25', 1.00), ('2004-06-30', 1.25),
    ('2004-08-10', 1.50), ('2004-09-21', 1.75), ('2004-11-10', 2.00),
    ('2004-12-14', 2.25), ('2005-02-02', 2.50), ('2005-03-22', 2.75),
    ('2005-05-03', 3.00), ('2005-06-30', 3.25), ('2005-08-09', 3.50),
    ('2005-09-20', 3.75), ('2005-11-01', 4.00), ('2005-12-13', 4.25),
    ('2006-01-31', 4.50), ('2006-03-28', 4.75), ('2006-05-10', 5.00),
    ('2006-06-29', 5.25), ('2007-09-18', 4.75), ('2007-10-31', 4.50),
    ('2007-12-11', 4.25), ('2008-01-22', 3.50), ('2008-01-30', 3.00),
    ('2008-03-18', 2.25), ('2008-04-30', 2.00), ('2008-10-08', 1.50),
    ('2008-10-29', 1.00), ('2008-12-16', 0.125),
    ('2015-12-17', 0.375), ('2016-12-15', 0.625), ('2017-03-16', 0.875),
    ('2017-06-15', 1.125), ('2017-12-14', 1.375), ('2018-03-22', 1.625),
    ('2018-06-14', 1.875), ('2018-09-27', 2.125), ('2018-12-20', 2.375),
    ('2019-08-01', 2.125), ('2019-09-19', 1.875), ('2019-10-31', 1.625),
    ('2020-03-04', 1.125), ('2020-03-16', 0.125),
    ('2022-03-17', 0.375), ('2022-05-05', 0.875), ('2022-06-16', 1.625),
    ('2022-07-28', 2.375), ('2022-09-22', 3.125), ('2022-11-03', 3.875),
    ('2022-12-15', 4.375), ('2023-02-02', 4.625), ('2023-03-23', 4.875),
    ('2023-05-04', 5.125), ('2023-07-27', 5.375), ('2024-09-19', 4.875),
    ('2024-11-08', 4.625), ('2024-12-19', 4.375),
]
fomc_decisions = pd.DataFrame(FOMC_DECISIONS, columns=['date','rate'])
fomc_decisions['date'] = pd.to_datetime(fomc_decisions['date'])
fomc_decisions = fomc_decisions.set_index('date').sort_index()
fomc_rate = fomc_decisions.reindex(
    pd.date_range(fomc_decisions.index.min(), pd.Timestamp.today(), freq='D')
).ffill().rename_axis('date')

# add rate series data from BoE

boe_raw = pd.read_csv('/content/drive/MyDrive/FED/BOEBR.csv')
boe_decisions = pd.DataFrame({
    'date': pd.to_datetime(boe_raw['Date Changed'], dayfirst=True),
    'rate': pd.to_numeric(boe_raw['Rate'], errors='coerce')
}).dropna().sort_values('date').set_index('date')
boe_rate = boe_decisions.reindex(
    pd.date_range(boe_decisions.index.min(), pd.Timestamp.today(), freq='D')
).ffill().rename_axis('date')
boe_rate.to_csv('/content/drive/MyDrive/FED/BOE_rate.csv')

# add rate series data from RBA

def parse_rate(s):
    s = str(s).strip()
    if 'to' in s: return float(s.split('to')[-1].strip())
    try: return float(s)
    except: return None

try:
    r = requests.get('https://www.rba.gov.au/statistics/tables/csv/a2-data.csv',
                     headers=HEADERS, timeout=15)
    assert r.status_code == 200
    df_raw  = pd.read_csv(StringIO(r.text), skiprows=4)
    df_data = df_raw.iloc[4:][['Type','Original.1']].copy()
    df_data.columns = ['date','rate_str']
    df_data['rate'] = df_data['rate_str'].apply(parse_rate)
    df_data['date'] = pd.to_datetime(df_data['date'], dayfirst=True)
    df_data = df_data.dropna(subset=['rate']).sort_values('date').set_index('date')
    print('RBA: live fetch successful')
except Exception as e:
    print(f'RBA live fetch failed ({e}), using hardcoded fallback')
    RBA_DECISIONS = [
        ('1990-01-23',17.50),('1994-08-10',5.50),('1999-11-03',5.00),
        ('2000-02-02',5.50),('2000-04-05',6.00),('2000-05-03',6.25),
        ('2001-02-07',5.75),('2001-03-07',5.50),('2001-04-04',5.25),
        ('2001-09-05',5.00),('2001-10-03',4.50),('2002-06-05',4.75),
        ('2003-11-05',5.00),('2005-03-02',5.25),('2006-05-03',5.75),
        ('2006-08-02',6.00),('2007-08-08',6.25),('2007-11-07',6.75),
        ('2008-02-05',7.00),('2008-03-04',7.25),('2008-09-03',7.00),
        ('2008-10-08',6.00),('2008-11-05',5.25),('2008-12-03',4.25),
        ('2009-02-04',3.25),('2009-04-08',3.00),('2009-10-07',3.25),
        ('2009-11-04',3.50),('2009-12-02',3.75),('2010-03-03',4.00),
        ('2010-04-07',4.25),('2010-05-05',4.50),('2010-11-03',4.75),
        ('2011-11-02',4.50),('2012-05-02',3.75),('2012-06-06',3.50),
        ('2012-10-03',3.25),('2012-12-05',3.00),('2013-05-08',2.75),
        ('2013-08-07',2.50),('2015-02-04',2.25),('2015-05-06',2.00),
        ('2016-05-04',1.75),('2016-08-03',1.50),('2019-06-05',1.25),
        ('2019-07-03',1.00),('2019-10-02',0.75),('2020-03-04',0.50),
        ('2020-03-19',0.25),('2020-11-04',0.10),('2022-05-04',0.35),
        ('2022-06-08',0.85),('2022-07-06',1.35),('2022-08-03',1.85),
        ('2022-09-07',2.35),('2022-10-05',2.60),('2022-11-02',2.85),
        ('2022-12-07',3.10),('2023-02-08',3.35),('2023-03-08',3.60),
        ('2023-05-03',3.85),('2023-06-07',4.10),('2023-11-08',4.35),
        ('2025-02-18',4.10),
    ]
    df_data = pd.DataFrame(RBA_DECISIONS, columns=['date','rate'])
    df_data['date'] = pd.to_datetime(df_data['date'])
    df_data = df_data.set_index('date').sort_index()

rba_rate = df_data[['rate']].reindex(
    pd.date_range(df_data.index.min(), pd.Timestamp.today(), freq='D')
).ffill().rename_axis('date')
rba_rate.to_csv('/content/drive/MyDrive/FED/RBA_rate.csv')

print(f'FOMC: {fomc_rate.index.min().date()} → {fomc_rate.index.max().date()}')
print(f'BoE:  {boe_rate.index.min().date()} → {boe_rate.index.max().date()}')
print(f'RBA:  {rba_rate.index.min().date()} → {rba_rate.index.max().date()}')

In [ ]:
# create binary labels used for training: change or hold, depending on rate decision

def get_rate_on_or_after(rate_df, meeting_date):
    """Rate effective the day after the meeting (decisions take effect next business day)."""
    next_day = pd.Timestamp(meeting_date) + pd.Timedelta(days=1)
    future   = rate_df[rate_df.index >= next_day]
    return future.iloc[0]['rate'] if len(future) else None

# assign labels based on rate change threshold, minimal value required to ensure actual rate change occurred

def assign_labels(meetings_df, rate_df, bank_name, threshold=0.10):
    df = meetings_df[meetings_df['bank'] == bank_name].copy()
    df = df.sort_values('meeting_date').reset_index(drop=True)
    df['rate_after']  = df['meeting_date'].apply(lambda d: get_rate_on_or_after(rate_df, d))
    df['rate_before'] = df['rate_after'].shift(1)
    df['rate_change'] = df['rate_after'] - df['rate_before']
    df['label']       = df['rate_change'].apply(
        lambda x: 'change' if pd.notna(x) and abs(x) > threshold else 'hold'
    )
    return df.dropna(subset=['rate_before'])

rate_map = {'FOMC': fomc_rate, 'BoE': boe_rate, 'RBA': rba_rate}
labeled  = pd.concat(
    [assign_labels(minutes_df, rate_df, bank) for bank, rate_df in rate_map.items()],
    ignore_index=True
)

print('Label distribution:')
print(labeled.groupby(['bank','label']).size().unstack(fill_value=0))

In [ ]:
# validate the labeled csv with several spot checks of critical rate change periods

checks = [
    ('FOMC', '2022-03-16', 'change')
    , ('FOMC', '2020-03-15', 'change')
    , ('FOMC', '2019-07-31', 'change')
    , ('FOMC', '2015-12-16', 'change')
    , ('FOMC', '2006-06-29', 'change')
    , ('RBA',  '2022-05-03', 'change')
]
print('Spot checks:')
for bank, date, expected in checks:
    row = labeled[
        (labeled['bank'] == bank) &
        (labeled['meeting_date'].astype(str).str.startswith(date))
    ]
    if len(row) == 0:
        print(f'  ? {bank} {date}: NOT FOUND')
    else:
        got    = row.iloc[0]['label']
        chg    = row.iloc[0]['rate_change']
        status = 'PASS' if got == expected else 'FAIL'
        print(f'  {status} {bank} {date}: {got} (change={chg:+.3f})')

out_path = '/content/drive/MyDrive/central_bank_minutes_labeled.csv'
labeled.to_csv(out_path, index=False)
print(f'\nSaved {len(labeled)} rows → {out_path}')